In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [5]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from tqdm import tqdm

# ==============================================================
# 1. YOUR EXACT KAGGLE DATASET PATHS (Updated based on your search)
# ==============================================================
ham_csv = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv'

ham_folders = [
    '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1', 
    '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2',
    # Some Kaggle versions dump all images into one folder. We include it just in case:
    '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/ham10000_images_part_1',
    '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/ham10000_images_part_2'
]

isic_root = '/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic/skin cancer isic the international skin imaging collaboration/Skin cancer ISIC The International Skin Imaging Collaboration'

# Output master directory in Kaggle's writable space
base_dir = '/kaggle/working/unified_dataset'

# Class Mappings (Unifying both datasets into 8 specific classes)
ham_mapping = {'nv': 'melanocytic_nevi', 'mel': 'melanoma', 'bcc': 'basal_cell_carcinoma', 'akiec': 'actinic_keratosis', 'bkl': 'benign_keratosis', 'df': 'dermatofibroma', 'vasc': 'vascular_lesion'}
isic_mapping = {'actinic keratosis': 'actinic_keratosis', 'basal cell carcinoma': 'basal_cell_carcinoma', 'dermatofibroma': 'dermatofibroma', 'melanoma': 'melanoma', 'nevus': 'melanocytic_nevi', 'pigmented benign keratosis': 'benign_keratosis', 'seborrheic keratosis': 'benign_keratosis', 'squamous cell carcinoma': 'squamous_cell_carcinoma', 'vascular lesion': 'vascular_lesion'}

# ==============================================================
# 2. CREATE THE DIRECTORIES
# ==============================================================
classes = set(ham_mapping.values()).union(set(isic_mapping.values()))
for split in ['train', 'val', 'test']:
    for class_name in classes:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

# ==============================================================
# 3. SPLIT HAM10000 (Protecting Lesion IDs from Data Leakage)
# ==============================================================
print("Processing HAM10000 Dataset...")
metadata = pd.read_csv(ham_csv)
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(gss_test.split(metadata, groups=metadata['lesion_id']))
df_train_val, df_test = metadata.iloc[train_val_idx], metadata.iloc[test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val['lesion_id']))
df_train, df_val = df_train_val.iloc[train_idx], df_train_val.iloc[val_idx]

def move_ham(df, split_name):
    missing_count = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Linking HAM {split_name}"):
        label = ham_mapping[row['dx']]
        img_name = row['image_id'] + '.jpg'
        found = False
        
        for folder in ham_folders:
            src = os.path.join(folder, img_name)
            if os.path.exists(src):
                # Symlinks are instant and use 0 extra disk space!
                dst = os.path.join(base_dir, split_name, label, img_name)
                if not os.path.exists(dst):
                    os.symlink(src, dst)
                found = True
                break
        if not found:
            missing_count += 1
            
    if missing_count > 0:
        print(f"Warning: {missing_count} images were not found in the provided folders.")

move_ham(df_train, 'train')
move_ham(df_val, 'val')
move_ham(df_test, 'test')

# ==============================================================
# 4. SPLIT ISIC 9 CLASSES DATASET
# ==============================================================
print("\nProcessing ISIC 9 Classes Dataset...")
for split_folder in ['Train', 'Test']:
    split_path = os.path.join(isic_root, split_folder)
    if not os.path.exists(split_path): continue

    for folder_name in os.listdir(split_path):
        if folder_name not in isic_mapping: continue
        target_label = isic_mapping[folder_name]
        src_folder = os.path.join(split_path, folder_name)
        images = [f for f in os.listdir(src_folder) if f.endswith('.jpg') or f.endswith('.png')]
        
        if split_folder == 'Train':
            # Split ISIC Train into Train (90%) and Val (10%)
            train_imgs, val_imgs = train_test_split(images, test_size=0.10, random_state=42)
            
            for img in train_imgs: 
                dst = os.path.join(base_dir, 'train', target_label, img)
                if not os.path.exists(dst): os.symlink(os.path.join(src_folder, img), dst)
                    
            for img in val_imgs: 
                dst = os.path.join(base_dir, 'val', target_label, img)
                if not os.path.exists(dst): os.symlink(os.path.join(src_folder, img), dst)
                    
        elif split_folder == 'Test':
            for img in images: 
                dst = os.path.join(base_dir, 'test', target_label, img)
                if not os.path.exists(dst): os.symlink(os.path.join(src_folder, img), dst)

print("\nAll data is unified and linked.")

Processing HAM10000 Dataset...


Linking HAM test: 100%|██████████| 2024/2024 [00:02<00:00, 792.61it/s]



Processing ISIC 9 Classes Dataset...

All data is unified and linked.


In [6]:
import cv2
import numpy as np
import torch
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler


class AdvancedHairRemoval(object):

    def __call__(self, img):
        # Convert PIL image to OpenCV NumPy array (RGB to BGR)
        cv_img = np.array(img)[:, :, ::-1].copy()
        
        # Convert to grayscale to find dark hair contrasts
        gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
        
        # 1. Blackhat Transform to isolate hair strands
        # A 17x17 rectangular kernel is standard for detecting thin hair-like structures
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
        blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
        
        # 2. Create a mask of the hairs
        # Threshold: values > 10 become 255 (pure white mask for hairs)
        _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
        
        # 3. Image Inpainting using Fast Marching Method (TELEA)
        # Replaces the hair mask with surrounding skin color
        inpainted_img = cv2.inpaint(cv_img, mask, 3, cv2.INPAINT_TELEA)
        
        # Convert back to RGB and then back to PIL Image for PyTorch
        inpainted_img = inpainted_img[:, :, ::-1]
        return Image.fromarray(inpainted_img)


IMG_SIZE = 600
BATCH_SIZE = 16 # Assuming you are using T4 x2 GPUs

# 3.1: Hair removal -> Resize -> Rotate -> Zoom -> Flip
train_transforms = transforms.Compose([
    AdvancedHairRemoval(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=45),                # Rotation
    transforms.RandomAffine(degrees=0, scale=(0.8, 1.2)), # Zooming
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Test data also needs hair removed and resized, but NO augmentation
val_transforms = transforms.Compose([
    AdvancedHairRemoval(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train_dataset = datasets.ImageFolder(root='/kaggle/working/unified_dataset/train', transform=train_transforms)
val_dataset = datasets.ImageFolder(root='/kaggle/working/unified_dataset/val', transform=val_transforms)

# Apply weights to solve Class Imbalance without creating fake images on disk
class_counts = [0] * len(train_dataset.classes)
for _, label in train_dataset.samples:
    class_counts[label] += 1
class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
sample_weights = [class_weights[label] for _, label in train_dataset.samples]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# Using num_workers=4 here to handle the heavy OpenCV CPU math
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print("Section 3.1 Pipeline Done")

Section 3.1 Pipeline Done


In [7]:
import torch
import torch.nn as nn
from torchvision import models

# CELL 3: EFFICIENTNET MODEL ARCHITECTURE (Sections 3.2 & 3.4)

def build_efficientnet_b7_paper(num_classes=8):

    # 1. Load EfficientNet-B7 with ImageNet weights (Section 3.2)
    # This imports the pre-trained weights to act as a feature extractor.
    weights = models.EfficientNet_B7_Weights.IMAGENET1K_V1
    model = models.efficientnet_b7(weights=weights)
    
    # 2. Freeze the base layers for Phase 1 of Transfer Learning
    # We do not want to destroy the ImageNet weights during initial training.
    for param in model.parameters():
        param.requires_grad = False
        
    # 3. Apply Section 3.4 Modifications for B7
    # The paper replaces the default head with a Dropout + Linear layer for 8 classes.
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_features, num_classes)
    )
    
    return model

# Initialize the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate the model architecture
model_b7 = build_efficientnet_b7_paper(num_classes=8)

# LEVERAGE KAGGLE'S MULTI-GPU SETTING (T4 x2)

# EfficientNet-B7 has 66 million parameters. We split the math across both GPUs!
if torch.cuda.device_count() > 1:
    print(f"Success: Detected {torch.cuda.device_count()} GPUs. Enabling DataParallel.")
    model_b7 = nn.DataParallel(model_b7)

model_b7 = model_b7.to(device)

print(f"EfficientNet-B7 Architecture successfully built and moved to: {device}")

Downloading: "https://download.pytorch.org/models/efficientnet_b7_lukemelas-c5b4e57e.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b7_lukemelas-c5b4e57e.pth


100%|██████████| 255M/255M [00:01<00:00, 217MB/s] 


Success: Detected 2 GPUs. Enabling DataParallel.
EfficientNet-B7 Architecture successfully built and moved to: cuda


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import copy

# 1. EARLY STOPPING UTILITY 

class EarlyStopping:
    def __init__(self, patience=3, delta=0.0):
        self.patience = patience
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False
        self.delta = delta
        self.best_model_wts = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            # If using DataParallel, save the underlying module's dict
            if isinstance(model, nn.DataParallel):
                self.best_model_wts = copy.deepcopy(model.module.state_dict())
            else:
                self.best_model_wts = copy.deepcopy(model.state_dict())
            self.counter = 0
            print("  [+] Validation Loss Improved. Model weights saved!")
        else:
            self.counter += 1
            print(f"  [-] No improvement. EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

# 2. TRAINING & EVALUATION FUNCTIONS
def run_epoch(model, dataloader, optimizer, criterion, is_train):
    if is_train:
        model.train()
    else:
        model.eval()
        
    running_loss, correct, total = 0.0, 0, 0
    
    # Using tqdm for a clean progress bar
    for inputs, labels in tqdm(dataloader, leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        
        if is_train:
            optimizer.zero_grad()
            
        with torch.set_grad_enabled(is_train):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            if is_train:
                loss.backward()
                optimizer.step()
                
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    return running_loss / total, correct / total

# 3. PHASE 1: FIXED FEATURE EXTRACTOR (Train Head Only)
print("SECTION 3.3 / PHASE 1: Training Classification Head")
criterion = nn.CrossEntropyLoss()

# Only pass the parameters of the classifier head to the optimizer
if isinstance(model_b7, nn.DataParallel):
    classifier_params = model_b7.module.classifier.parameters()
else:
    classifier_params = model_b7.classifier.parameters()

optimizer_phase1 = optim.Adam(classifier_params, lr=0.001)

EPOCHS_PHASE_1 = 5
for epoch in range(EPOCHS_PHASE_1):
    train_loss, train_acc = run_epoch(model_b7, train_loader, optimizer_phase1, criterion, is_train=True)
    val_loss, val_acc = run_epoch(model_b7, val_loader, optimizer_phase1, criterion, is_train=False)
    print(f"Phase 1 | Epoch {epoch+1}/{EPOCHS_PHASE_1} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Val Loss: {val_loss:.4f}")

# 4. PHASE 2: FINE-TUNING (Unfreeze Top Layers)
print("\nSECTION 3.3 / PHASE 2: Fine-Tuning Top Layers")

# Unfreeze the last 4 blocks for fine-tuning
target_model = model_b7.module if isinstance(model_b7, nn.DataParallel) else model_b7
for param in target_model.features[-4:].parameters():
    param.requires_grad = True

# Optimizer now handles all layers that require gradients (LR drops to 0.0001)
optimizer_phase2 = optim.Adam(filter(lambda p: p.requires_grad, target_model.parameters()), lr=0.0001)

# Polynomial Decay Scheduler & Early Stopping
EPOCHS_PHASE_2 = 15
polynomial_decay = lambda epoch: (1 - (epoch / EPOCHS_PHASE_2)) ** 0.9 
scheduler_phase2 = optim.lr_scheduler.LambdaLR(optimizer_phase2, lr_lambda=polynomial_decay)
early_stopper = EarlyStopping(patience=3)

for epoch in range(EPOCHS_PHASE_2):
    train_loss, train_acc = run_epoch(model_b7, train_loader, optimizer_phase2, criterion, is_train=True)
    val_loss, val_acc = run_epoch(model_b7, val_loader, optimizer_phase2, criterion, is_train=False)
    
    print(f"Phase 2 | Epoch {epoch+1}/{EPOCHS_PHASE_2} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Val Loss: {val_loss:.4f}")
    
    scheduler_phase2.step()
    early_stopper(val_loss, model_b7)
    
    if early_stopper.early_stop:
        print("Early stopping triggered! Halted to prevent overfitting to the medical domain.")
        break

# 5. SAVE BEST WEIGHTS

# Save the optimal weights found during fine-tuning
torch.save(early_stopper.best_model_wts, '/kaggle/working/best_efficientnet_b7.pth')
print("\nTransfer Learning Complete! Best fine-tuned model saved to disk.")

SECTION 3.3 / PHASE 1: Training Classification Head


Phase 1 | Epoch 1/5 | Train Acc: 45.49% | Val Acc: 59.56% | Val Loss: 1.2291


Phase 1 | Epoch 2/5 | Train Acc: 52.11% | Val Acc: 58.46% | Val Loss: 1.2298


Phase 1 | Epoch 3/5 | Train Acc: 53.68% | Val Acc: 60.06% | Val Loss: 1.1493


Phase 1 | Epoch 4/5 | Train Acc: 54.11% | Val Acc: 60.96% | Val Loss: 1.1164


Phase 1 | Epoch 5/5 | Train Acc: 54.58% | Val Acc: 61.36% | Val Loss: 1.0866

SECTION 3.3 / PHASE 2: Fine-Tuning Top Layers


Phase 2 | Epoch 1/15 | Train Acc: 63.78% | Val Acc: 72.57% | Val Loss: 0.7685
  [+] Validation Loss Improved. Model weights saved!


Phase 2 | Epoch 2/15 | Train Acc: 73.68% | Val Acc: 73.77% | Val Loss: 0.6720
  [+] Validation Loss Improved. Model weights saved!


Phase 2 | Epoch 3/15 | Train Acc: 76.33% | Val Acc: 69.57% | Val Loss: 0.8436
  [-] No improvement. EarlyStopping counter: 1 out of 3


 43%|████▎     | 226/529 [05:23<07:04,  1.40s/it]